# batchnorm-affine-params — ex2: recover BatchNorm gamma and beta from before-after pairs

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `batchnorm-affine-params`. Running the final beacon cell reports progress against the `CNN: BatchNorm affine params` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: BatchNorm affine params` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`batchnorm-affine-params`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "batchnorm-affine-params"
DD_SUBTOPIC = "CNN: BatchNorm affine params"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Recovering BN's gamma/beta from before-after pairs — quick refresher

BatchNorm's affine step is `y[:, c] = gamma[c] * x_hat[:, c] + beta[c]` per channel. Given access to both `x_hat` and `y` (the inputs and outputs of the affine step), you can recover `(gamma, beta)` per channel via simple linear regression — but for the BN case there's a much cleaner trick.

**Two-point recovery.** Pick any TWO entries within the same channel where `x_hat` has distinct values:

```
y0 = gamma * x_hat0 + beta
y1 = gamma * x_hat1 + beta

gamma = (y1 - y0) / (x_hat1 - x_hat0)        # slope
beta  = y0 - gamma * x_hat0                  # y-intercept
```

**Per-channel form.** Because the affine is channel-independent, you can do all `C` channels in parallel:

```
# Flatten everything per channel, then pick two distinct (x_hat, y) pairs.
x_hat_flat = einops.rearrange(x_hat, 'b c h w -> c (b h w)')
y_flat     = einops.rearrange(y,     'b c h w -> c (b h w)')
gamma = (y_flat[:, 1] - y_flat[:, 0]) / (x_hat_flat[:, 1] - x_hat_flat[:, 0])
beta  = y_flat[:, 0] - gamma * x_hat_flat[:, 0]
```

**Why this works.** The affine map `x → gamma * x + beta` is an **affine line** in (x_hat, y) space — slope `gamma`, intercept `beta`. Two points determine a line, so two `(x_hat, y)` pairs determine `(gamma, beta)` exactly.

### Exercise 2 — recover BatchNorm gamma and beta from before-after pairs

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze BatchNorm's per-channel affine map `y = gamma * x_hat + beta` by recovering `gamma` and `beta` from two distinct `(x_hat, y)` pairs per channel — verifying the affine line is fully determined by two points.
> Keywords: batchnorm, affine, inverse, two-point-recovery
> ```

**KCs targeted:** `bn-affine-line-recovery`, `bn-per-channel-independence`

Implement `ex2_recover_bn_params(x_hat, y)`. Given:

- `x_hat: (B, C, H, W)` — pre-affine normalized input.
- `y: (B, C, H, W)` — post-affine output (you know `y = gamma * x_hat + beta` was applied per channel).

Return `(gamma, beta)` — each a 1-D tensor of length `C` — recovered from the data.

**Two-point recovery, per channel.**

For each channel `c`, pick two **distinct** entries within that channel where `x_hat` has different values. Let them be `(x0, y0)` and `(x1, y1)`. Then:

```
gamma[c] = (y1 - y0) / (x1 - x0)        # slope of the affine line
beta[c]  = y0 - gamma[c] * x0           # y-intercept
```

**The cleanest vectorized form.** Flatten each channel and take the first two entries (you can assume the test inputs have distinct `x_hat[:, c, 0, 0]` and `x_hat[:, c, 0, 1]` for every `c`):

```
x_hat_flat = einops.rearrange(x_hat, 'b c h w -> c (b h w)')
y_flat     = einops.rearrange(y,     'b c h w -> c (b h w)')
gamma = (y_flat[:, 1] - y_flat[:, 0]) / (x_hat_flat[:, 1] - x_hat_flat[:, 0])
beta  = y_flat[:, 0] - gamma * x_hat_flat[:, 0]
```

**Why this works.** An affine map in 1-D is a LINE; two distinct points uniquely determine a line's slope and intercept. Since BN's affine is channel-INDEPENDENT, you can do all `C` channels in parallel (no cross-channel coupling).

The test runs you against a known `(gamma, beta)` and confirms your recovered values match to fp tolerance.

In [ ]:
def ex2_recover_bn_params(x_hat: Tensor, y: Tensor):
    """Return (gamma, beta) recovered from (x_hat, y) per channel."""
    raise NotImplementedError()


def _test_ex2():
    rng = t.Generator().manual_seed(0)

    # --- Known gamma/beta, generate y, recover ---
    B, C, H, W = 4, 5, 6, 6
    x_hat = t.randn(B, C, H, W, generator=rng)
    gamma_true = t.tensor([1.0, 2.5, -0.5, 0.0001, 7.3])
    beta_true  = t.tensor([0.0, -1.0, 3.14, 0.0, -2.7])
    y = gamma_true.view(1, -1, 1, 1) * x_hat + beta_true.view(1, -1, 1, 1)

    gamma_rec, beta_rec = ex2_recover_bn_params(x_hat, y)

    # --- Shape contract ---
    assert gamma_rec.shape == (C,), f'gamma shape: {tuple(gamma_rec.shape)}'
    assert beta_rec.shape  == (C,), f'beta shape: {tuple(beta_rec.shape)}'

    # --- Value recovery ---
    assert t.allclose(gamma_rec, gamma_true, atol=1e-5), (
        f'gamma mismatch:\n  recovered {gamma_rec}\n  expected  {gamma_true}'
    )
    assert t.allclose(beta_rec, beta_true, atol=1e-5), (
        f'beta mismatch:\n  recovered {beta_rec}\n  expected  {beta_true}'
    )

    # --- Round-trip: applying recovered params reconstructs y ---
    y_recon = gamma_rec.view(1, -1, 1, 1) * x_hat + beta_rec.view(1, -1, 1, 1)
    assert t.allclose(y_recon, y, atol=1e-5), 'gamma_rec, beta_rec must reconstruct y'

    # --- Identity case: y = x_hat → gamma = ones, beta = zeros ---
    x2 = t.randn(2, 4, 3, 3, generator=rng)
    y2 = x2.clone()
    g2, b2 = ex2_recover_bn_params(x2, y2)
    assert t.allclose(g2, t.ones(4), atol=1e-5), f'identity gamma should be 1: got {g2}'
    assert t.allclose(b2, t.zeros(4), atol=1e-5), f'identity beta should be 0: got {b2}'

    # --- Pure shift: gamma = 1, beta = something ---
    x3 = t.randn(1, 3, 4, 4, generator=rng)
    shift = t.tensor([5.0, -10.0, 0.5])
    y3 = x3 + shift.view(1, -1, 1, 1)
    g3, b3 = ex2_recover_bn_params(x3, y3)
    assert t.allclose(g3, t.ones(3), atol=1e-5)
    assert t.allclose(b3, shift, atol=1e-5)

    # --- Pure scale: gamma = something, beta = 0 ---
    x4 = t.randn(1, 3, 4, 4, generator=rng)
    scale = t.tensor([2.0, -1.5, 0.25])
    y4 = scale.view(1, -1, 1, 1) * x4
    g4, b4 = ex2_recover_bn_params(x4, y4)
    assert t.allclose(g4, scale, atol=1e-5)
    assert t.allclose(b4, t.zeros(3), atol=1e-5)

    # --- Large C (parallel recovery across many channels) ---
    C_big = 32
    x5 = t.randn(2, C_big, 5, 5, generator=rng)
    g5_true = t.randn(C_big, generator=rng)
    b5_true = t.randn(C_big, generator=rng)
    y5 = g5_true.view(1, -1, 1, 1) * x5 + b5_true.view(1, -1, 1, 1)
    g5, b5 = ex2_recover_bn_params(x5, y5)
    assert t.allclose(g5, g5_true, atol=1e-5)
    assert t.allclose(b5, b5_true, atol=1e-5)
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_recover_bn_params(x_hat: Tensor, y: Tensor):
    x_flat = einops.rearrange(x_hat, 'b c h w -> c (b h w)')
    y_flat = einops.rearrange(y,     'b c h w -> c (b h w)')
    # Two distinct points per channel — first two entries of the flat axis.
    x0 = x_flat[:, 0]
    x1 = x_flat[:, 1]
    y0 = y_flat[:, 0]
    y1 = y_flat[:, 1]
    gamma = (y1 - y0) / (x1 - x0)
    beta  = y0 - gamma * x0
    return gamma, beta
```

**Why two points suffice.** An affine map `y = gamma * x + beta` has exactly TWO free parameters (slope and intercept). Two distinct `(x, y)` pairs give two equations in those two unknowns — uniquely solvable. Any THIRD point must lie on the recovered line (a hidden invariant the test could check).

**Why the rearrange pattern.** `'b c h w -> c (b h w)'` flattens the (B, H, W) axes per channel while keeping C as the leading axis. The result `(C, B*H*W)` lets us index `[:, 0]` and `[:, 1]` to get two entries per channel simultaneously — all `C` recoveries happen as a vectorized computation.

**A more robust alternative** (out of scope here) would be least-squares regression per channel: fit `(gamma, beta)` to ALL entries in that channel via `gamma, beta = lstsq([x, 1], y)`. Two-point is faster but amplifies noise if `x0` and `x1` happen to be very close. For exact arithmetic (our test case), both produce identical results.

**Channel independence is what makes this work.** If BN had cross-channel coupling (like `y[c] = gamma[c, c'] * x[c'] + beta[c]`), you couldn't recover params channel-by-channel — you'd need joint regression over all `C` channels. The strict per-channel affine is what enables the trivial recovery.

**The connection back to BN's forward.** This drill exercises INVERTING the affine. The forward `gamma * x_hat + beta` and this inverse are duals — together they show that the affine stage is a bijection (modulo `gamma = 0` channels, which collapse to a constant).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()